# 실습 6-2 : STUCCO (대조집합 학습)

#### **<실습 내용>**

1. 대조집합 학습 개요
- STUCCO 알고리즘 원리

2. STUCCO 기본 예제 (Process_Data)
- 데이터 전처리 (연속형 변수 범주화)
- 대조집합(cset) 도출 및 해석

3. Vibe Coding 실습 (통신사 고객 이탈 데이터)

## 분석 준비

### 주요 라이브러리 호출

In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import logging
from stucco import ContrastSetLearner  # 대조집합 학습(Contrast Set Learning)을 수행하는 STUCCO 알고리즘 구현체

logging.getLogger().setLevel(logging.WARNING)  # stucco.py의 상세 진행 로그 숨기기

---

## 1) 데이터 불러오기

In [33]:
data = pd.read_csv("dataset/day6-2_data.csv")
print("데이터 크기:", data.shape)
data.head()

데이터 크기: (596, 12)


,Prod_ID,M1,M2,P1,P2,P3,C1,C2,P4,P5,P6,Y
0,1001001,15,E,A,B,D,110.192,27.049,A,A,B,0
1,1001002,20,E,A,B,D,109.940,27.215,A,A,A,0
2,1001003,15,U,A,B,B,110.214,27.444,B,B,B,4
3,1001004,25,K,B,B,C,107.260,27.300,A,A,C,0
4,1001005,20,E,A,A,D,105.026,28.283,B,B,A,0


In [34]:
# 불량 기준 설정: 제품 1개당 불량 개수(Y)가 3개 이상이면 Fault, 아니면 Pass로 그룹 라벨 생성
data["Fail"] = ["Fault" if i >= 3 else "Pass" for i in data["Y"]]
data

,Prod_ID,M1,M2,P1,P2,P3,C1,C2,P4,P5,P6,Y,Fail
0,1001001,15,E,A,B,D,110.192,27.049,A,A,B,0,Pass
1,1001002,20,E,A,B,D,109.940,27.215,A,A,A,0,Pass
2,1001003,15,U,A,B,B,110.214,27.444,B,B,B,4,Fault
3,1001004,25,K,B,B,C,107.260,27.300,A,A,C,0,Pass
4,1001005,20,E,A,A,D,105.026,28.283,B,B,A,0,Pass
...,...,...,...,...,...,...,...,...,...,...,...,...,...
591,1001592,15,E,A,C,D,115.153,31.652,A,A,C,0,Pass
592,1001593,15,E,A,B,D,114.593,33.890,A,A,B,0,Pass
593,1001594,20,U,A,A,B,105.724,33.251,B,B,A,0,Pass
594,1001595,25,E,A,A,A,109.783,32.054,A,A,D,0,Pass


In [35]:
print("클래스 분포:")
print(data["Fail"].value_counts())

클래스 분포:
Fail
Pass     471
Fault    125
Name: count, dtype: int64


**활용할 반도체 공정 데이터 소개 (6-1과 동일)**

| 변수 | 설명 |
|---|---|
| Prod_ID | 제품 고유 ID |
| M1 | 공정 설비 온도 설정값 (15/20/25/30) |
| M2 | 사용 설비 ID (E, U, K) |
| P1 ~ P6 | 제품 설계/공정 옵션 (범주형 파라미터) |
| C1, C2 | 공정 중 측정된 연속형 값 |
| Y | 제품 1개당 발생한 불량 개수 |
| Fail | Y >= 3이면 Fault, 아니면 Pass로 정의한 라벨 |

## 2) 전처리하기

> STUCCO는 입력변수로 **범주형 변수만** 허용하므로 연속형 변수(C1, C2)는 구간화하여 범주형 변수 처럼 나타내야함

In [36]:
# 1. M1
# 숫자(15/20/25/30)로 저장되어 있음
# 그러나 실제로는 '온도 설정값'을 의미하는 범주이므로 범주형으로 변환
data["M1"] = data["M1"].astype("object")

# 2. 연속형 변수 C1, C2
# qcut: 데이터 개수가 균등하도록 4구간으로 분할
data["C1_cut"] = pd.qcut(data["C1"], 4)
# cut: 의미 있는 경계값 기준으로 직접 3구간 지정
data["C2_cut"] = pd.cut(data["C2"], bins=[18, 28, 32, 39], labels=["Low", "Normal", "High"]) 

### 2-1) STUCCO 실행

1. 비교할 그룹 설정
2. Contrast Set 후보 규칙 생성
3. 유의미한 규칙 추출

In [37]:
# STUCCO에 불필요한 변수 제거
# (ID, 이미 구간화한 원본 연속형 변수, 라벨 만드는 데 쓴 Y)
contra_data = data.drop(["Prod_ID", "C1", "C2", "Y"], axis=1)
contra_data

,M1,M2,P1,P2,P3,P4,P5,P6,Fail,C1_cut,C2_cut
0,15,E,A,B,D,A,A,B,Pass,"(109.191, 112.225]",Low
1,20,E,A,B,D,A,A,A,Pass,"(109.191, 112.225]",Low
2,15,U,A,B,B,B,B,B,Fault,"(109.191, 112.225]",Low
3,25,K,B,B,C,A,A,C,Pass,"(106.365, 109.191]",Low
4,20,E,A,A,D,B,B,A,Pass,"(102.158, 106.365]",Normal
...,...,...,...,...,...,...,...,...,...,...,...
591,15,E,A,C,D,A,A,C,Pass,"(112.225, 123.34]",Normal
592,15,E,A,B,D,A,A,B,Pass,"(112.225, 123.34]",High
593,20,U,A,A,B,B,B,A,Pass,"(102.158, 106.365]",High
594,25,E,A,A,A,A,A,D,Pass,"(109.191, 112.225]",High


> 1. 비교할 그룹 설정 (ContrastSetLearner())

In [38]:
# ContrastSetLearner(DataFrame, group_feature)

# "Fail" 컬럼의 그룹을 기준으로 Contrast Set 탐색
# Contrast Set: 그룹 간 차이를 가장 잘 설명하는 규칙

learner = ContrastSetLearner(contra_data, group_feature="Fail")
learner

> 2. Contrast Set 후보 규칙 생성 (learn())

In [39]:
# learner.learn():
# 그룹 간 차이를 설명할 수 있는 규칙 후보(조건 조합)를 생성
# 각 규칙의 그룹별 등장 빈도를 계산

# max_length: 규칙을 구성하는 최대 조건 수
# n_matrices: 규칙이 각 그룹에서 몇 번 나타났는지 등의 빈도 정보를 저장

n_matrices = learner.learn(max_length=3)
n_matrices

4217

> 3. 유의미한 규칙 추출 (score())

In [40]:
# learner.score():
# learn( )에서 생성한 빈도 정보를 활용해 규칙별 Support, Confidence, Lift를 계산

# Support: 규칙의 출현 비율
# Confidence: 규칙이 나타났을 때 해당 그룹일 확률
# Lift: 특정 그룹과의 연관성 정도

# lift가 2.0 이상인(특정 그룹과의 연관성 정도) 규칙만 채택
contrast_rules = learner.score(min_lift=2.0)
contrast_rules

,rule,group,lift
4,"(P1=>A, P3=>A, C2_cut=>High)",Fail=>Fault,2.666667
13,"(M2=>U, P2=>A, P3=>A)",Fail=>Fault,2.650104
7,"(M2=>K, P1=>A, P6=>D)",Fail=>Fault,2.625000
15,"(M1=>25, P1=>A, P3=>C)",Fail=>Fault,2.300000
14,"(M1=>25, P1=>A, C2_cut=>Low)",Fail=>Fault,2.300000
5,"(P1=>A, P6=>D, C1_cut=>(106.365, 109.191])",Fail=>Fault,2.258065
9,"(M2=>K, P1=>A, P2=>B)",Fail=>Fault,2.250000
10,"(P1=>A, P5=>B, C1_cut=>(112.225, 123.34])",Fail=>Fault,2.222222
8,"(P1=>A, P3=>D, P6=>D)",Fail=>Fault,2.181818
16,"(M1=>25, M2=>K, P1=>A)",Fail=>Fault,2.181818


In [41]:
# group 컬럼 값이 "Fail=>Fault", "Fail=>Pass" 형태로 저장되어 있음
# 보기 쉽게 "Fault", "Pass"만 남도록 문자열 변경
contrast_rules["group"] = contrast_rules["group"].str.replace("Fail=>", "")

# learn() 단계에서 생성된 전체 조건 조합(규칙 후보) 수 출력
print("생성된 조건 조합 수:", n_matrices)

# score() 단계의 기준을 통과하여 최종 선택된 Contrast Set 규칙 수 출력
print("도출된 규칙 수:", contrast_rules.shape[0])

생성된 조건 조합 수: 4217
도출된 규칙 수: 17


### 2-2) 결과 해석

In [42]:
# 도출된 규칙 확인
# 실제로는 Pass 규칙도 있었는데 score 기준에 따라 필터링 된 것임

contrast_rules.head()

,rule,group,lift
4,"(P1=>A, P3=>A, C2_cut=>High)",Fault,2.666667
13,"(M2=>U, P2=>A, P3=>A)",Fault,2.650104
7,"(M2=>K, P1=>A, P6=>D)",Fault,2.625000
15,"(M1=>25, P1=>A, P3=>C)",Fault,2.300000
14,"(M1=>25, P1=>A, C2_cut=>Low)",Fault,2.300000


In [43]:
# Fault 그룹을 설명하는 규칙만 추출
fault_rules = contrast_rules.loc[contrast_rules["group"] == "Fault"]

# Lift 기준 상위 5개 규칙 확인
fault_rules.sort_values(by="lift", ascending=False).head()

,rule,group,lift
4,"(P1=>A, P3=>A, C2_cut=>High)",Fault,2.666667
13,"(M2=>U, P2=>A, P3=>A)",Fault,2.650104
7,"(M2=>K, P1=>A, P6=>D)",Fault,2.625000
15,"(M1=>25, P1=>A, P3=>C)",Fault,2.300000
14,"(M1=>25, P1=>A, C2_cut=>Low)",Fault,2.300000


In [44]:
# Lift가 가장 높은 Contrast Set 규칙
# (P1=A) AND (P3=A) AND (C2_cut=High)를 만족하는 데이터 선택

cset_mask = (data["P1"] == "A") & (data["P3"] == "A") & (data["C2_cut"] == "High")

# 규칙을 만족하는 데이터의 불량(Fail) 분포 확인
print("cset 해당 데이터 불량 분포:")
print(data.loc[cset_mask, "Fail"].value_counts())
print()

# 규칙을 만족하지 않는 데이터의 불량(Fail) 분포 확인
print("cset 미해당 데이터 불량 분포:")
print(data.loc[~cset_mask, "Fail"].value_counts())

cset 해당 데이터 불량 분포:
Fail
Fault    12
Pass      6
Name: count, dtype: int64

cset 미해당 데이터 불량 분포:
Fail
Pass     465
Fault    113
Name: count, dtype: int64


---

## 3) Vibe Coding 실습 (통신사 고객 이탈 데이터)

STUCCO를 **통신사 고객 이탈(Customer Churn) 데이터셋**에 직접 적용해 봅니다.

- `dataset/day6_vibecoding.csv` 파일 사용하기
- 타겟 변수: `Churn` (No: 유지, Yes: 이탈)

In [46]:
# 필요한 라이브러리 호출하기
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer

# 0. 데이터 불러오기
data = pd.read_csv('dataset/day6_vibecoding.csv')

# 1. 결측치 처리
# 범주형(InternetService, PaymentMethod) → 최빈값 대치
for col in ['InternetService', 'PaymentMethod']:
    data[col] = data[col].fillna(data[col].mode()[0])

# TotalCharges → 중앙값 대치
data['TotalCharges'] = data['TotalCharges'].fillna(data['TotalCharges'].median())

# tenure, MonthlyCharges → KNN Imputer (비슷한 고객들의 값을 참고해 대치)
knn_cols = ['tenure', 'MonthlyCharges']
imputer = KNNImputer(n_neighbors=5)
data[knn_cols] = imputer.fit_transform(data[knn_cols])

# 2. 군집화에 쓰지 않을 컬럼 분리 
# Churn: 군집화 단계에서는 사용하지 않고, 군집 해석(군집별 이탈률 비교) 시에만 참고
churn = data['Churn']
X = data.drop(columns=['Churn'])

# 3. 수치형 변수 스케일링 
# K-Means/DBSCAN은 거리 기반이라 변수 스케일이 다르면 특정 변수가 거리 계산을 왜곡할 수 있음
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

# 4. 범주형 변수 인코딩 
# 값이 2개뿐인 이진 범주형 변수는 0/1로 Label Encoding
binary_map = {
    'gender':           {'Male': 1, 'Female': 0},
    'Partner':          {'Yes': 1, 'No': 0},
    'Dependents':       {'Yes': 1, 'No': 0},
    'PhoneService':     {'Yes': 1, 'No': 0},
    'PaperlessBilling': {'Yes': 1, 'No': 0},
}
for col, mapping in binary_map.items():
    X[col] = X[col].map(mapping)

# 값이 3개 이상인 다중 범주형 변수는 One-Hot Encoding
ohe_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity',
            'OnlineBackup', 'DeviceProtection', 'TechSupport',
            'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']
X = pd.get_dummies(X, columns=ohe_cols, drop_first=True)

X.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,MultipleLines_No phone service,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,-1.286412,0,1,-1.173697,-0.994242,True,...,False,False,False,False,False,False,False,False,True,False
1,1,0,0,0,0.069187,1,0,-0.265601,-0.173244,False,...,False,False,False,False,False,True,False,False,False,True
2,1,0,0,0,-1.245333,1,1,-0.369479,-0.959674,False,...,False,False,False,False,False,False,False,False,False,True
3,1,0,0,0,0.521054,0,0,-0.756509,-0.194766,True,...,True,False,False,False,False,True,False,False,False,False
4,0,0,0,0,-1.245333,1,1,0.195149,-0.940470,False,...,False,False,False,False,False,False,False,False,True,False


**[과제]** 본 데이터셋에서의 연속형 범수인 'tenure', 'MonthlyCharges', 'TotalCharges'를 `pd.qcut` 등으로 구간화하고 `ContrastSetLearner`로 Churn(Yes/No) 그룹 간 대조집합을 도출하는 코드를 AI와 상의해서 작성하세요.

In [47]:
# 0. 데이터 다시 불러오기 (셀 30의 X는 군집화용 스케일링/원핫인코딩이 적용되어 STUCCO에 부적합)
data = pd.read_csv('dataset/day6_vibecoding.csv')

# 1. 결측치 처리 (셀 30과 동일한 방식)
for col in ['InternetService', 'PaymentMethod']:
    data[col] = data[col].fillna(data[col].mode()[0])
data['TotalCharges'] = data['TotalCharges'].fillna(data['TotalCharges'].median())

knn_cols = ['tenure', 'MonthlyCharges']
imputer = KNNImputer(n_neighbors=5)
data[knn_cols] = imputer.fit_transform(data[knn_cols])

# 2. SeniorCitizen: 0/1 정수 → 범주형으로 변환
#    (숫자로 두면 stucco가 빈도 편향이 큰 수치형으로 판단해 자동 제거함)
data['SeniorCitizen'] = data['SeniorCitizen'].astype(str)

# 3. 연속형 변수 구간화
# tenure: 고객 생애주기 관점의 의미 있는 경계로 직접 분할 (cut)
data['tenure_cut'] = pd.cut(
    data['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['0-1yr', '1-2yr', '2-4yr', '4yr+'],
    include_lowest=True,
)

# MonthlyCharges: 4분위수 기반 균등 분할 (qcut)
data['MonthlyCharges_cut'] = pd.qcut(
    data['MonthlyCharges'], 4,
    labels=['Low', 'Mid-Low', 'Mid-High', 'High'],
)

# TotalCharges: 오른쪽 치우침이 심해 등간격(cut) 대신 qcut 사용
data['TotalCharges_cut'] = pd.qcut(
    data['TotalCharges'], 4,
    labels=['Low', 'Mid-Low', 'Mid-High', 'High'],
)

# 4. 구간화 결과 확인 (구간별 개수 + 구간별 이탈률)
for col in ['tenure_cut', 'MonthlyCharges_cut', 'TotalCharges_cut']:
    summary = data.groupby(col, observed=True)['Churn'].value_counts(normalize=True).unstack()
    print(f"--- {col} ---")
    print(pd.concat([data[col].value_counts().sort_index().rename('count'),
                     summary['Yes'].rename('churn_rate')], axis=1))
    print()

--- tenure_cut ---
            count  churn_rate
tenure_cut                   
0-1yr        2156    0.470315
1-2yr        1027    0.288218
2-4yr        1659    0.207957
4yr+         2201    0.097229

--- MonthlyCharges_cut ---
                    count  churn_rate
MonthlyCharges_cut                   
Low                  1761    0.120386
Mid-Low              1764    0.252268
Mid-High             1757    0.366534
High                 1761    0.322544

--- TotalCharges_cut ---
                  count  churn_rate
TotalCharges_cut                   
Low                1761    0.433844
Mid-Low            1766    0.251982
Mid-High           1755    0.230769
High               1761    0.144804



In [50]:
contra_data = data.drop(['tenure', 'MonthlyCharges', 'TotalCharges', 'TotalCharges'], axis=1)

In [51]:
# 1. STUCCO 입력 데이터 구성
# 구간화 전 원본 연속형 변수 3개만 제거 (Churn과 *_cut 컬럼, 나머지 범주형은 유지)
contra_data = data.drop(['tenure', 'MonthlyCharges', 'TotalCharges'], axis=1)

# 2. 비교할 그룹 설정
# copy()를 넘기는 이유: ContrastSetLearner가 내부에서 DataFrame을 직접 수정하기 때문에
# (컬럼 값을 "컬럼명=>값" 문자열로 바꿈) 원본을 보호해야 재실행이 가능함
learner = ContrastSetLearner(contra_data.copy(), group_feature='Churn')

# 3. Contrast Set 후보 규칙 생성
# 변수가 17개라 max_length=3이면 조합 수가 급증해 오래 걸릴 수 있음 → 2로 시작 권장
n_matrices = learner.learn(max_length=2)
print("생성된 조건 조합 수:", n_matrices)

# 4. 유의미한 규칙 추출
contrast_rules = learner.score(min_lift=2.0)
contrast_rules['group'] = contrast_rules['group'].str.replace('Churn=>', '')
print("도출된 규칙 수:", contrast_rules.shape[0])
contrast_rules.head(10)

생성된 조건 조합 수: 1380
도출된 규칙 수: 15


,rule,group,lift
14,"(Contract=>Month-to-month, MonthlyCharges_cut=...",Yes,3.012694
13,"(InternetService=>Fiber optic, tenure_cut=>0-1yr)",Yes,2.991705
11,"(Contract=>Month-to-month, MonthlyCharges_cut=...",Yes,2.486783
1,"(TechSupport=>No, tenure_cut=>0-1yr)",Yes,2.368571
0,"(OnlineSecurity=>No, tenure_cut=>0-1yr)",Yes,2.362314
10,"(DeviceProtection=>Yes, Contract=>Month-to-month)",Yes,2.304293
3,"(Contract=>Month-to-month, TotalCharges_cut=>Low)",Yes,2.292527
12,"(OnlineBackup=>Yes, Contract=>Month-to-month)",Yes,2.178756
8,"(StreamingTV=>Yes, Contract=>Month-to-month)",Yes,2.167145
9,"(StreamingMovies=>Yes, Contract=>Month-to-month)",Yes,2.162906


**[과제]** 도출된 규칙 중 이탈(Yes) 그룹에서 lift가 높은 상위 규칙들을 확인하고 어떤 조건 조합이 이탈과 강하게 연관되는지 AI와 함께 해석해 보세요.

규칙들을 관통하는 4가지 패턴
1. Month-to-month 계약이 이탈의 핵심 축 (10개 중 7개 규칙에 등장). 월 단위 계약은 해지 비용(위약금, 잔여 약정)이 없어서 불만이 생기면 바로 떠날 수 있는 상태입니다. 거의 모든 규칙이 "Month-to-month + 무언가"의 형태인 것은, 월 단위 계약이 이탈의 전제 조건이고 나머지 조건이 이탈 확률을 증폭시키는 구조임을 보여줍니다.

2. 가입 1년 미만 신규 고객 (규칙 13, 1, 0). tenure_cut=>0-1yr이 3개 규칙에 등장합니다. 신규 고객은 서비스에 대한 관성(lock-in)이 없고, 초기 경험이 기대에 못 미치면 빠르게 이탈합니다. 특히 13번 **"Fiber optic + 가입 1년 미만" (lift 2.99, 이탈률 ~79%)**이 눈에 띄는데, 광랜은 요금이 비싼 상품이라 "비싼 요금을 내기 시작한 지 얼마 안 된 고객이 가격 대비 만족을 못 느끼면 바로 떠난다"는 전형적인 조기 이탈 패턴입니다.

3. 높은 월 요금 (규칙 14, 11, 3). 1위 규칙인 "Month-to-month + 월요금 High" (lift 3.01, 이탈률 ~80%)는 **"비싼 요금 + 언제든 나갈 수 있는 계약"**의 조합입니다. 3번 규칙의 TotalCharges_cut=>Low도 같은 맥락인데, 누적 요금이 낮다는 건 곧 가입 기간이 짧다는 뜻이라(TotalCharges ≈ tenure × MonthlyCharges) 사실상 "신규 + 월 단위 계약" 패턴의 변형입니다.

4. 안전망 서비스의 부재 (규칙 1, 0). TechSupport=>No, OnlineSecurity=>No가 신규 고객과 결합했습니다. 기술지원이나 보안 서비스는 문제가 생겼을 때 회사와의 접점을 만들어주고 서비스 의존도를 높이는 역할을 하는데, 이게 없는 고객은 묶어두는 요소가 하나도 없는 상태입니다.

**[과제]** 가장 중요한 규칙에 해당하는 고객들의 실제 이탈률을 비교해서 해당 규칙이 실제로 유의미한지 검증해 보세요.

In [52]:
cset_mask = (data['Contract'] == 'Month-to-month') & (data['MonthlyCharges_cut'] == 'High')

print("규칙 해당 고객의 이탈 분포:")
print(data.loc[cset_mask, 'Churn'].value_counts(normalize=True))
print("\n규칙 미해당 고객의 이탈 분포:")
print(data.loc[~cset_mask, 'Churn'].value_counts(normalize=True))

규칙 해당 고객의 이탈 분포:
Churn
Yes    0.518002
No     0.481998
Name: proportion, dtype: float64

규칙 미해당 고객의 이탈 분포:
Churn
No     0.769816
Yes    0.230184
Name: proportion, dtype: float64


검증 결과가 잘 나왔고, 결론부터 말하면 이 규칙은 실제로 유의미합니다. 규칙 해당 고객의 이탈률은 51.8%로 미해당 고객(23.0%)의 약 2.25배, 전체 평균(26.5%)의 약 2배입니다. 고객 2명 중 1명이 이탈하는 집단을 특정해낸 것이니 대조집합으로서 충분히 의미가 있습니다.

다만 제가 앞서 "lift 3.0이면 이탈률 약 80%"라고 추정했는데 실제로는 51.8%가 나왔죠. 이 차이는 규칙이 잘못된 게 아니라 stucco가 lift를 계산하는 방식이 일반적인 정의와 다르기 때문입니다. 설명하면:

방금 하신 검증은 "규칙 해당 vs 나머지 전부"의 비교입니다. 일반적인 lift 개념이죠.
반면 stucco.py의 score()는 규칙의 대조군을 만들 때 get_rule_negations()를 사용하는데, 이 함수는 규칙의 모든 조건이 동시에 부정된 조합만 대조군으로 삼습니다. 즉 "(Month-to-month, High)"의 대조군은 "(월 단위 계약이 아니면서 요금도 High가 아닌)" 고객뿐이고, "Month-to-month인데 요금은 낮은" 고객처럼 조건이 하나만 어긋나는 경우는 비교에서 아예 빠집니다.
그런데 "연 단위 계약 + 낮은 요금" 고객은 이탈률이 극히 낮은 집단이라, 이들만을 대조군으로 삼으면 lift가 실제(전체 대비)보다 크게 나옵니다. 그래서 stucco 내부 lift는 3.01이지만, 전체 모집단 기준으로 다시 재면 51.8 / 26.5 ≈ 1.95가 되는 겁니다. stucco의 lift는 규칙 간 상대적 순위를 매기는 지표로 쓰고, 실제 이탈률 수치는 지금 하신 것처럼 마스크 검증으로 확인하는 것이 올바른 사용법입니다.

검증을 마무리하는 차원에서 두 가지를 더 확인해 보면 좋습니다.

In [53]:
# 1. 규칙에 해당하는 고객 규모 (너무 소수 집단이면 실용성이 떨어짐)
print("규칙 해당 고객 수:", cset_mask.sum(), "/", len(data),
      f"({cset_mask.mean():.1%})")

# 2. 카이제곱 검정으로 통계적 유의성 확인
from scipy.stats import chi2_contingency
ct = pd.crosstab(cset_mask, data['Churn'])
chi2, p_value, dof, expected = chi2_contingency(ct)
print(f"chi2 = {chi2:.1f}, p-value = {p_value:.2e}")

규칙 해당 고객 수: 861 / 7043 (12.2%)
chi2 = 319.7, p-value = 1.72e-71


이탈률 차이가 51.8% vs 23.0%로 크고 표본도 수천 건이라 p-value는 사실상 0에 가깝게 나올 겁니다. 여기까지 하면 셀 35 과제의 요구사항(규칙의 실제 유의미성 검증)은 완결됩니다.

정리하면: "Month-to-month 계약 + 높은 월 요금" 고객은 전체 평균의 2배인 절반 이상이 이탈하는 고위험군이며, 통계적으로도 유의합니다. 실무적으로는 이 집단에 연 단위 계약 전환 할인이나 요금제 재설계를 우선 적용하는 근거로 쓸 수 있습니다.